In [1]:
import os
import torch
import torch.utils.data as data

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: idx for idx, aa in enumerate(AMINO_ACIDS)}

class SequenceFileDataset(data.Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.target_transform = target_transform
        self.samples = []
        self.classes = []
        self.class_to_idx = {}

        filenames = sorted(
            [f for f in os.listdir(root_dir) if os.path.isfile(os.path.join(root_dir, f))]
        )

        for idx, fname in enumerate(filenames):
            self.classes.append(fname)
            self.class_to_idx[fname] = idx
            filepath = os.path.join(root_dir, fname)

            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    seq = line.strip()
                    if seq:
                        self.samples.append((seq, idx))

        self.num_classes = len(self.classes)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sequence, label = self.samples[idx]
        if self.transform is not None:
            sequence = self.transform(sequence)
        if self.target_transform is not None:
            label = self.target_transform(label)
        return sequence, label
    

def transform_sequence(seq):
    if isinstance(seq, bytes):
        seq = seq.decode("utf-8")

    seq = seq.strip().upper()
    if len(seq) != 9:
        raise ValueError(f"Sequence length must be 9, got {len(seq)}: {seq}")

    one_hot = torch.zeros(len(seq), len(AMINO_ACIDS), dtype=torch.float32)
    for i, aa in enumerate(seq):
        if aa in AA_TO_INDEX:
            one_hot[i, AA_TO_INDEX[aa]] = 1.0

    return one_hot.view(-1)


from collections import defaultdict
import random

def stratified_train_test_split(dataset, test_size=0.1, shuffle=True, seed=42):
    labels = [label for _, label in dataset.samples]
    label_to_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        label_to_indices[label].append(idx)

    train_indices = []
    test_indices = []
    rng = random.Random(seed)

    for label, inds in label_to_indices.items():
        if shuffle:
            rng.shuffle(inds)
        if len(inds) == 1:
            n_test = 0
        else:
            n_test = int(round(len(inds) * test_size))
            n_test = max(1, min(n_test, len(inds) - 1))

        test_indices.extend(inds[:n_test])
        train_indices.extend(inds[n_test:])

    return train_indices, test_indices


def encode_label_one_hot(label, num_classes):
    one_hot = torch.zeros(num_classes, dtype=torch.float32)
    one_hot[label] = 1.0
    return one_hot

def encode_label_pos(label, num_classes):
    one_hot = torch.zeros(1, dtype=torch.float32)
    if label + 1 >= num_classes:
        one_hot = torch.zeros(1, dtype=torch.float32)
    return one_hot

# Example usage
root = "ex1_data"
dataset = SequenceFileDataset(
    root,
    transform=transform_sequence,
)
dataset.target_transform = lambda label: encode_label_one_hot(label, dataset.num_classes)

train_indices, test_indices = stratified_train_test_split(dataset, test_size=0.1)
train_dataset = data.Subset(dataset, train_indices)
test_dataset = data.Subset(dataset, test_indices)
train_dataloader = data.DataLoader(train_dataset, batch_size=512, shuffle=True)

print("classes:", dataset.classes)
print("class_to_idx:", dataset.class_to_idx)
print("total samples:", len(dataset))
print("first sample shape:", dataset[0][0].shape)
print("first label shape:", dataset[0][1].shape)


classes: ['A0101_pos.txt', 'A0201_pos.txt', 'A0203_pos.txt', 'A0207_pos.txt', 'A0301_pos.txt', 'A2402_pos.txt', 'negs.txt']
class_to_idx: {'A0101_pos.txt': 0, 'A0201_pos.txt': 1, 'A0203_pos.txt': 2, 'A0207_pos.txt': 3, 'A0301_pos.txt': 4, 'A2402_pos.txt': 5, 'negs.txt': 6}
total samples: 37383
first sample shape: torch.Size([180])
first label shape: torch.Size([7])


In [2]:

import torch.nn as nn

class SimpleClassifier(nn.Module):

    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        # Initialize the modules we need to build the network
        self.linear1 = nn.Linear(in_features=num_inputs, out_features=128)
        self.linear2 = nn.Linear(in_features=128, out_features=64)
        self.linear3 = nn.Linear(in_features=64, out_features=num_outputs)
        self.act_fn = nn.ReLU()

    def forward(self, x):
        # Perform the calculation of the model to determine the prediction
        x = self.linear1(x)
        x = self.act_fn(x)
        x = self.linear2(x)
        x = self.act_fn(x)
        x = self.linear3(x)
        return x
    


model = SimpleClassifier(num_inputs=180, num_outputs=7)
loss_module = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [3]:
import torch
import torch.utils.data as data
import numpy as np
from tqdm.auto import tqdm
import plotly.graph_objects as go
from IPython.display import display

def train_model_epoch_eval(
    model,
    optimizer,
    train_dataset,
    test_dataset,
    loss_module,
    num_epochs=150,
    batch_size=128,
    device="cpu"
):
    """
    Trains 'model' for 'num_epochs' and evaluates after each epoch on the entire test set in one pass.
    Uses a single pass for test data (assuming test_dataset is small).
    """
    # 1) Set up DataLoaders
    train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # If the test set is tiny, we can put the entire dataset into a single batch
    test_loader = data.DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

    train_losses = []
    test_losses = []
    model.to(device)

    def is_pos_label(label_idx):
        label_name = train_dataset.dataset.classes[label_idx]
        return "pos" in label_name.lower()

    fig = go.FigureWidget()
    fig.add_scatter(name="Train loss", mode="lines+markers", x=[], y=[] )
    fig.add_scatter(name="Test loss", mode="lines+markers", x=[], y=[] )
    fig.update_layout(
        title="Loss over epochs",
        xaxis_title="Epoch",
        yaxis_title="Loss",
        legend_title="Loss type",
    )
    display(fig)

    for epoch in tqdm(range(num_epochs)):
        ####################################################
        # Training phase
        ####################################################
        model.train()
        epoch_train_loss = []

        for data_inputs, data_labels in train_loader:
            # 1) Move input data and labels to device
            data_inputs = data_inputs.to(device)
            data_labels = data_labels.to(device)

            # 2) Forward pass
            preds = model(data_inputs)

            # 3) Compute loss
            loss = loss_module(preds, data_labels)
            epoch_train_loss.append(loss.item())

            # 4) Backprop + weight update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Average loss for this epoch
        train_losses.append(np.mean(epoch_train_loss))

        ####################################################
        # Evaluation phase (single pass over the entire test set)
        ####################################################
        with torch.no_grad():
            model.eval()

            # Get the single batch from test_loader
            (X_test, Y_test) = next(iter(test_loader))
            X_test = X_test.to(device)
            Y_test = Y_test.to(device)

            # Forward pass
            test_preds = model(X_test)
            test_probs = torch.softmax(test_preds, dim=1)

            # Compute test loss
            test_loss = loss_module(test_preds, Y_test)
            test_losses.append(test_loss.item())

            pred_label_idxs = test_probs.argmax(dim=1)
            true_label_idxs = Y_test.argmax(dim=1)
            pred_pos_neg = torch.tensor([1 if is_pos_label(int(idx)) else 0 for idx in pred_label_idxs], device=device)
            true_pos_neg = torch.tensor([1 if is_pos_label(int(idx)) else 0 for idx in true_label_idxs], device=device)
            pos_neg_acc = (pred_pos_neg == true_pos_neg).float().mean().item()

        fig.data[0].x = list(range(1, len(train_losses) + 1))
        fig.data[0].y = train_losses
        fig.data[1].x = list(range(1, len(test_losses) + 1))
        fig.data[1].y = test_losses

        print(f"Epoch {epoch+1}/{num_epochs} test_loss={test_loss.item():.4f} pos_neg_acc={pos_neg_acc:.3f}")

    display(fig)
    print("Training complete.")

train_model_epoch_eval(
    model,
    optimizer,
    train_dataset,
    test_dataset,
    loss_module,
)


FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Train loss',
              'type': 'scatter',
              'uid': '81727d0d-3fef-4d2a-abe8-bbb405818129',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Test loss',
              'type': 'scatter',
              'uid': 'c6a83da5-248f-495b-bd93-8233400c1e3b',
              'x': [],
              'y': []}],
    'layout': {'legend': {'title': {'text': 'Loss type'}},
               'template': '...',
               'title': {'text': 'Loss over epochs'},
               'xaxis': {'title': {'text': 'Epoch'}},
               'yaxis': {'title': {'text': 'Loss'}}}
})

  0%|          | 0/150 [00:00<?, ?it/s]

/home/tabibon/deep_learning/ex1/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1/150 test_loss=0.2783 pos_neg_acc=0.655
Epoch 2/150 test_loss=0.2727 pos_neg_acc=0.655
Epoch 3/150 test_loss=0.2679 pos_neg_acc=0.655


KeyboardInterrupt: 

In [7]:
a

tensor([[-2.7449, -4.6486, -4.2667,  ..., -2.7342, -3.3904,  1.9311],
        [-2.4095, -3.2905, -3.4891,  ..., -3.0022, -3.3018,  0.4595],
        [-2.7200, -2.8677, -2.8491,  ..., -2.5217, -2.5709,  0.3833],
        ...,
        [-2.3808, -3.8899, -3.7226,  ..., -2.6480, -3.3154,  1.3573],
        [-4.2033, -4.5727, -4.1009,  ..., -3.5842, -0.4368,  0.7824],
        [-2.1801, -2.2586, -2.8012,  ..., -2.8977, -4.3875, -0.2333]])

In [8]:
b

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.]])

In [9]:
test_loss = loss_module(a, b.float())

In [10]:
test_loss

tensor(0.2042)

In [ ]:
a[:3]

tensor([[-2.7449, -4.6486, -4.2667, -4.1233, -2.7342, -3.3904,  1.9311],
        [-2.4095, -3.2905, -3.4891, -2.1924, -3.0022, -3.3018,  0.4595],
        [-2.7200, -2.8677, -2.8491, -2.8978, -2.5217, -2.5709,  0.3833]])

In [12]:
b[:3]

tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0.]])

In [13]:
loss_module(a[:3], b[:3].float())

tensor(0.6054)